# Week 16, Day 5: Towers of Hanoi and flood fill

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w016d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def countdown(n):
    if n == 0:
        return
    countdown(n - 1)
    print(n, end=" ")
    countdown(n - 1)

countdown(3)

In [ ]:
def hanoi(n, source, target, spare):
    if n == 0:
        return
    hanoi(n - 1, source, spare, target)
    print(f"move disc {n} from {source} to {target}")
    hanoi(n - 1, spare, target, source)

hanoi(3, "A", "C", "B")

*Interactive figure on the website: Step through the solution. Change the number of discs: each extra disc doubles the moves, plus one..*

In [ ]:
def moves(n):
    return 0 if n == 0 else 2 * moves(n - 1) + 1

for n in (1, 2, 3, 10, 20, 64):
    print(n, moves(n))

In [ ]:
def hanoi_moves(n, source="A", target="C", spare="B"):
    if n == 0:
        return []
    return (hanoi_moves(n - 1, source, spare, target)
            + [(source, target)]
            + hanoi_moves(n - 1, spare, target, source))

print(hanoi_moves(2))
print(len(hanoi_moves(10)))

> **Key idea.** Hanoi: to move n discs, move n − 1 out of the way, move the biggest, then move the n − 1 back on top. Trust the recursion to handle the smaller towers. It takes 2ⁿ − 1 moves.

In [ ]:
grid = ["..#",
        ".##",
        "..."]

def neighbours(grid, r, c):
    found = []
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(grid) and 0 <= nc < len(grid[0]):
            found.append((nr, nc))
    return found

print(neighbours(grid, 0, 0))
print(neighbours(grid, 1, 1))

In [ ]:
def fill(grid, r, c):
    if r < 0 or r >= len(grid) or c < 0 or c >= len(grid[0]):
        return
    if grid[r][c] != ".":
        return              # a wall, or filled already
    grid[r][c] = "~"
    fill(grid, r - 1, c)
    fill(grid, r + 1, c)
    fill(grid, r, c - 1)
    fill(grid, r, c + 1)

picture = [list(row) for row in ["....#...",
                                 ".##.#...",
                                 ".#..####",
                                 ".#......"]]
fill(picture, 0, 0)
for row in picture:
    print("".join(row))

*Interactive figure on the website: Each number is the order that square was filled. The fill goes as far as it can in one direction before trying the next, and the open calls show the path back..*

In [ ]:
def region(grid, r, c):
    seen = set()
    kind = grid[r][c]
    def go(r, c):
        if not (0 <= r < len(grid) and 0 <= c < len(grid[0])):
            return
        if grid[r][c] != kind or (r, c) in seen:
            return
        seen.add((r, c))
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            go(r + dr, c + dc)
    go(r, c)
    return seen

grid = ["aab",
        "abb",
        "ccb"]
print(sorted(region(grid, 0, 0)))
print(len(region(grid, 0, 2)))

In [ ]:
def count_islands(grid):
    rows, cols = len(grid), len(grid[0])
    seen = set()
    def sink(r, c):
        if not (0 <= r < rows and 0 <= c < cols) or grid[r][c] != "#" or (r, c) in seen:
            return 0
        seen.add((r, c))
        return 1 + sink(r - 1, c) + sink(r + 1, c) + sink(r, c - 1) + sink(r, c + 1)
    sizes = []
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == "#" and (r, c) not in seen:
                sizes.append(sink(r, c))
    return sizes

world = ["##....#",
         "#...###",
         "...#...",
         "#..##..",
         "##....#"]
sizes = count_islands(world)
print(len(sizes), "islands, sizes", sizes)

In [ ]:
def solve(maze, r, c, path, seen):
    if not (0 <= r < len(maze) and 0 <= c < len(maze[0])):
        return False
    if maze[r][c] == "#" or (r, c) in seen:
        return False
    path.append((r, c))
    seen.add((r, c))
    if maze[r][c] == "E":
        return True
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        if solve(maze, r + dr, c + dc, path, seen):
            return True
    path.pop()               # dead end: step back
    return False

maze = ["S.#.....",
        ".##.###.",
        "....#...",
        "###.#.#.",
        "....#.#E"]
path = []
solve(maze, 0, 0, path, set())
print(len(path), "squares:", path)
for r, row in enumerate(maze):
    print("".join("*" if (r, c) in path and ch == "." else ch for c, ch in enumerate(row)))

> **Common mistake.** Recursive flood fill makes one nested call per square in a long winding region, so on a big grid (say 200 by 200) it can hit Python's recursion limit. The fix is the same idea with a loop and your own stack: keep a list of squares still to visit, pop one, and push its neighbours. The challenge in Phase 3 will use this. For the grids today, recursion is fine.

### Exercise: Hanoi as a list

Write `hanoi(n, source, target, spare)` that **returns** the list of moves as `(from, to)` tuples, using recursion. `hanoi(2, "A", "C", "B")` is `[("A", "B"), ("A", "C"), ("B", "C")]`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert hanoi(0, \"A\", \"C\", \"B\") == []\n    assert hanoi(1, \"A\", \"C\", \"B\") == [(\"A\", \"C\")]\n    assert hanoi(2, \"A\", \"C\", \"B\") == [(\"A\", \"B\"), (\"A\", \"C\"), (\"B\", \"C\")]\n    assert len(hanoi(10, \"A\", \"C\", \"B\")) == 1023\n    assert hanoi(3, \"L\", \"R\", \"M\")[3] == (\"L\", \"R\")")

### Exercise: Check a solution

Write `valid(n, moves)` that plays a list of `(from, to)` moves on pegs `"A"`, `"B"`, `"C"` starting with discs `n` (bottom) down to `1` (top) on `"A"`, and returns `True` only if every move is legal (the peg moved from isn't empty, and the disc never lands on a smaller one) and every disc ends on `"C"`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert valid(2, [(\"A\", \"B\"), (\"A\", \"C\"), (\"B\", \"C\")]) is True\n    assert valid(2, [(\"A\", \"C\"), (\"A\", \"C\")]) is False\n    assert valid(1, [(\"B\", \"C\")]) is False\n    assert valid(2, [(\"A\", \"B\"), (\"A\", \"C\")]) is False\n    assert valid(0, []) is True\n    solution = [(\"A\", \"C\"), (\"A\", \"B\"), (\"C\", \"B\"), (\"A\", \"C\"), (\"B\", \"A\"), (\"B\", \"C\"), (\"A\", \"C\")]\n    assert valid(3, solution) is True")

### Exercise: Paint bucket

Write `paint(picture, r, c, colour)` that takes a picture as a list of strings, and returns a **new** list of strings in which the square at `(r, c)` and every square connected to it (up, down, left, right) of the **same** character as it has been changed to `colour`. If that square is already `colour`, return the picture unchanged.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    pic = [\"aab\",\n           \"abb\",\n           \"bba\"]\n    assert paint(pic, 0, 0, \"x\") == [\"xxb\", \"xbb\", \"bba\"]\n    assert paint(pic, 0, 2, \"z\") == [\"aaz\", \"azz\", \"zza\"]\n    assert paint(pic, 2, 2, \"q\") == [\"aab\", \"abb\", \"bbq\"]\n    assert paint(pic, 0, 0, \"a\") == pic\n    assert pic == [\"aab\", \"abb\", \"bba\"]")

### Exercise: Lakes

On a map, `"~"` is water and `"."` is land. Write `lakes(grid)` that returns a list of the sizes of the separate bodies of water (connected up, down, left and right), largest first.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    m = [\"~~..~\",\n         \"~...~\",\n         \"..~..\",\n         \"~~~.~\"]\n    assert lakes(m) == [4, 3, 2, 1]\n    assert lakes([\"..\", \"..\"]) == []\n    assert lakes([\"~~\", \"~~\"]) == [4]")

### Exercise: Can I get there?

Write `reachable(maze)` that returns `True` if there is a way through the maze (a list of strings) from the square `"S"` to the square `"E"`, moving up, down, left or right through anything except walls `"#"`. Find `"S"` yourself.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert reachable([\"S.#\",\n                      \"..#\",\n                      \"#.E\"]) is True\n    assert reachable([\"S#E\"]) is False\n    assert reachable([\"S...\",\n                      \"###.\",\n                      \"E#..\",\n                      \".#.#\"]) is False\n    assert reachable([\"E.S\"]) is True")

### Bug hunt: It never stops

This flood fill should count the open squares connected to the top corner (there are 6), but crashes with `RecursionError`. Fix it.

In [ ]:
grid = ["..#",
        ".##",
        "..."]

def count(r, c):
    if not (0 <= r < len(grid) and 0 <= c < len(grid[0])):
        return 0
    if grid[r][c] == "#":
        return 0
    return 1 + count(r - 1, c) + count(r + 1, c) + count(r, c - 1) + count(r, c + 1)

print(count(0, 0))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"six\"\n    expect_output(\"6\")")

### Bug hunt: Wrong way round

This Hanoi solution prints 3 moves for 2 discs, but they break the rules: the second move puts disc 2 on top of disc 1. Fix the one wrong line.

In [ ]:
def hanoi(n, source, target, spare):
    if n == 0:
        return
    hanoi(n - 1, source, target, spare)
    print(source, "->", target)
    hanoi(n - 1, spare, target, source)

hanoi(2, "A", "C", "B")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"legal moves\"\n    expect_output(\"A -> B\\nA -> C\\nB -> C\")")

### Puzzle: Region size

Arrange the lines for `size(grid, r, c, seen)`, which returns the number of `"#"` squares connected to `(r, c)`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Work these out without running them.

In [ ]:
a = ___   # how many moves to solve Hanoi with 5 discs?
b = ___   # in the 3-disc solution from A to C, what is the 4th move, as a tuple?
c = ___   # how many neighbours (4 directions) does the corner square of a grid have?
d = ___   # how many separate "#" regions in ["#.#", ".#.", "#.#"], counting up, down, left and right only?

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == 31 and b == (\"A\", \"C\") and c == 2 and d == 5")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: draw the path

Write `draw_path(maze)` that finds a path from `"S"` to `"E"` by backtracking, trying directions in the order up, down, left, right, and returns the maze as a list of strings with every `"."` on the path changed to `"*"` (`"S"` and `"E"` stay as they are). If there is no path, return the maze unchanged. Then write `count_paths(maze)` that returns how many different paths there are from `"S"` to `"E"` that never visit a square twice.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_draw():\n    \"draw\"\n    m = [\"S.#\",\n         \".##\",\n         \"..E\"]\n    assert draw_path(m) == [\"S.#\", \"*##\", \"**E\"]\n    assert draw_path([\"S#E\"]) == [\"S#E\"]\n    big = [\"S...#\",\n           \"##.#.\",\n           \"...#.\",\n           \".#...\",\n           \"...#E\"]\n    assert draw_path(big) == [\"S**.#\", \"##*#.\", \"..*#.\", \".#***\", \"...#E\"]\ndef test_count():\n    \"count\"\n    assert count_paths([\"S.\", \".E\"]) == 2\n    assert count_paths([\"S#E\"]) == 0\n    assert count_paths([\"S..\", \"...\", \"..E\"]) == 12")

## Warm-up

Two recursive calls with work in the middle: the pattern `1 2 1 3 1 2 1` has the 3 once, the 2 twice, the 1 four times. It's exactly the shape of today's first puzzle.

## The Towers of Hanoi

Three pegs, A, B and C. On peg A is a tower of discs, largest at the bottom. Move the whole tower to peg C, following two rules:

1. Move one disc at a time, taking the top disc from one peg and putting it on another.
2. Never put a disc on top of a smaller one.

Try planning it for four discs and you'll soon lose track. Now think recursively. To move a tower of n discs from A to C, the biggest disc must at some point move from A to C, and at that moment the other n − 1 discs must all be on B, out of the way. So:

1. Move the top n − 1 discs from A to B (using C as the spare).
2. Move the biggest disc from A to C.
3. Move the n − 1 discs from B to C (using A as the spare).

Steps 1 and 3 are smaller towers: the same problem, one disc fewer. A tower of 0 discs needs no moves at all.

Watch the moves on the pegs, and which call each belongs to:

How many moves for n discs? Moving n discs takes two moves of n − 1 discs, plus one: M(n) = 2 × M(n − 1) + 1, with M(0) = 0. That gives 1, 3, 7, 15, 31, ... which is **2ⁿ − 1**. Each extra disc doubles the work.

The puzzle was published in 1883 by the French mathematician Édouard Lucas, with a story of priests moving 64 golden discs, and the world ending when they finished. At one move a second, 2⁶⁴ − 1 moves take about 585 billion years. It's safe.

Returning the moves as a list, instead of printing them, makes the solution easy to test:

## Grids and neighbours

You met grids in Week 9: a list of rows, each a list (or string) of squares. `grid[r][c]` is the square in row `r`, column `c`. Its four **neighbours** (up, down, left, right) are one row or column away, but squares on the edges have fewer, so check the position is inside the grid before using it:

The list `[(-1, 0), (1, 0), (0, -1), (0, 1)]` is the standard way to write "the four directions". Adding the four diagonals gives eight neighbours, which some problems want.

## Flood fill

**Flood fill** starts at one square and spreads to every square connected to it by a path of squares of the same kind. Recursively: fill this square, then flood fill from each neighbour. The base cases are what make it stop:

- the position is outside the grid;
- the square is a wall (or a different colour);
- the square has **already been filled**.

The last one is essential. Without it, a square fills its neighbour, the neighbour fills the first square again, and so on for ever (until Python gives up with `RecursionError`).

Changing each filled square from `"."` to `"~"` does double duty: it's the result, **and** the mark that says "already done". The grid is a list of lists so the squares can be changed (strings can't be changed in place).

Click squares to see the order a recursive fill visits them in, and the calls still waiting:

When you don't want to change the grid, keep a `seen` set of positions instead:

## Counting regions

Count the separate regions (islands on a map, blobs in an image, groups of the same colour): look at every square, and each time you find one that isn't part of any region seen so far, that's a new region; flood fill it so its squares are counted as seen.

`sink` returns how many squares it filled: 1 for this square plus whatever each neighbour's fill finds. A filled-already or water square adds 0.

## Mazes

A maze is a grid with walls. Is there a way from the start to the exit? Flood fill from the start, and see whether the exit got filled. To get the actual **path**, use backtracking: step onto a square, try to finish from there, and if every direction fails, step back off it.

The path found isn't necessarily the **shortest**: it's the first one found by trying up, down, left, right in that order. Finding the shortest path needs a different search (breadth-first search), which comes in Phase 3 with graphs.

## Practice

## Quiz

## Daily challenge

## Before you go

Solve Hanoi with 4 discs on paper (or with coins of different sizes) by following the recursive plan, and check you used 15 moves. Draw a small maze of your own and run your solver on it. Tomorrow is the Forge: a project that puts the whole week's recursion to work.